#Deep Learning assignment2 - CNN's
###The work is dependent on the article following article: https://www.cs.cmu.edu/~rsalakhu/papers/oneshot1.pdf
###DataSet: https://talhassner.github.io/home/projects/lfwa/index.html.

## 1. Imports and Data Setup


In [ ]:
!pip install thop gdown

import os
import numpy as np
import matplotlib.pyplot as plt
import random
import urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.metrics import roc_curve, auc
from sklearn.manifold import TSNE
from thop import profile

# Official split files
!wget -q --no-check-certificate http://vis-www.cs.umass.edu/lfw/pairsDevTrain.txt
!wget -q --no-check-certificate http://vis-www.cs.umass.edu/lfw/pairsDevTest.txt

# Using gdown to download from a reliable Google Drive mirror of LFW-deepfunneled
import gdown
lfw_tar = "lfw-deepfunneled.tgz"
file_id = '1S96Y8pZ6aNf9m9pX-Y6N6LqXhX6-NnUe' # Public mirror ID for LFW

if not os.path.exists(lfw_tar) or os.path.getsize(lfw_tar) < 1000000:
    print("Downloading LFW-deepfunneled dataset via gdown...")
    url = f'https://drive.google.com/uc?id={file_id}'
    gdown.download(url, lfw_tar, quiet=False)

# Verify download and extract
if os.path.exists(lfw_tar) and os.path.getsize(lfw_tar) > 10000000:
    print("Extracting dataset...")
    !tar -xzf {lfw_tar}
    if os.path.exists("lfw-deepfunneled"):
        !mv lfw-deepfunneled lfw2
    elif os.path.exists("lfw"):
        !mv lfw lfw2
    print("Done.")
else:
    print("Download failed. Attempting direct wget from backup...")
    !wget -O {lfw_tar} "https://github.com/popcornell/LFW/raw/master/lfw-deepfunneled.tgz"

FileURLRetrievalError: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1S96Y8pZ6aNf9m9pX-Y6N6LqXhX6-NnUe

but Gdown can't. Please check connections and permissions.

In [ ]:
def parse_lfw_pairs(pairs_file, data_dir="lfw2"):
    """
    Parses the LFW pairs.txt format.
    Note: The extracted LFW-a folder is typically named 'lfw2'. Adjust data_dir if different.
    """
    with open(pairs_file, 'r') as f:
        lines = f.readlines()[1:] # Skip the first line (pair counts)

    pairs = []
    labels = []

    for line in lines:
        parts = line.strip().split()
        if len(parts) == 3:
            # Matched pair: name, id1, id2
            name, id1, id2 = parts
            img1 = os.path.join(data_dir, name, f"{name}_{int(id1):04d}.jpg")
            img2 = os.path.join(data_dir, name, f"{name}_{int(id2):04d}.jpg")
            pairs.append((img1, img2))
            labels.append(1.0) # 1 for same identity
        elif len(parts) == 4:
            # Mismatched pair: name1, id1, name2, id2
            name1, id1, name2, id2 = parts
            img1 = os.path.join(data_dir, name1, f"{name1}_{int(id1):04d}.jpg")
            img2 = os.path.join(data_dir, name2, f"{name2}_{int(id2):04d}.jpg")
            pairs.append((img1, img2))
            labels.append(0.0) # 0 for different identities

    return pairs, labels

# Load the official splits
all_train_pairs, all_train_labels = parse_lfw_pairs("pairsDevTrain.txt", data_dir="lfw2")
test_pairs, test_labels = parse_lfw_pairs("pairsDevTest.txt", data_dir="lfw2")

# Carve validation split from training pairs
# Justification for report: We perform a random 80/20 split on the paired data.
# While this means validation identities overlap with training identities, it provides
# a stable metric for model convergence on seen distributions, leaving pairsDevTest
# strictly for evaluating generalization to unseen identities as required.
val_ratio = 0.2
dataset_size = len(all_train_pairs)
indices = list(range(dataset_size))
random.seed(42)
random.shuffle(indices)

split_idx = int(np.floor(val_ratio * dataset_size))
val_idx, train_idx = indices[:split_idx], indices[split_idx:]

train_pairs = [all_train_pairs[i] for i in train_idx]
train_labels = [all_train_labels[i] for i in train_idx]

val_pairs = [all_train_pairs[i] for i in val_idx]
val_labels = [all_train_labels[i] for i in val_idx]

print(f"Total training pairs: {len(train_pairs)}")
print(f"Total validation pairs: {len(val_pairs)}")
print(f"Total testing pairs: {len(test_pairs)}")

## 2. Experiment 1 – Loss Function
Evaluation of three loss functions using a fixed Koch-style CNN backbone:
1. Koch-style (BCE)
2. Contrastive Loss
3. Triplet Loss with semi-hard negative mining

## 3. Experiment 2 – Backbone
Comparing architectural impacts using the best loss from Experiment 1:
- Koch-style CNN (reduced FC layer for parameter matching)
- ResNet-18 (trained from scratch)

## 4. Experiment 3 – Frozen Pretrained-Features Baseline
Baseline evaluation using a frozen ImageNet-pretrained ResNet-18.

## 5. Visualization and Comparison
Final analysis including ROC curves, accuracy tables, t-SNE projections, and failure case analysis.